# 반덤핑 품목별 HS 코드 변경 이력 테이블

**관련 이슈**: #7  
**목적**: 반덤핑 데이터의 유니크 품목별로 HS 개정판(H0~H6) 전환 시 코드가 어떻게 바뀌었는지 기록한 이력 테이블 생성  
**산출물**: `data/interim/반덤핑_HS코드_이력.csv`

---

## 테이블 구조

품목 1개당 HS 개정판(H0~H6) 7개 행을 가진 **long format** 구조.

```
품목명_정규화  | hs_revision | hs_code | hs_description | 변경여부 | 상태
폴리아세탈수지 | H0          | 3901    | Polymers...    | -        | needs_review
폴리아세탈수지 | H1          | 3901    | Polymers...    | False    | needs_review
폴리아세탈수지 | H2          | 3907    | Polyacetals... | True     | needs_review  ← 코드 변경!
...           | ...         | ...     | ...            | ...      | ...
```

새 품목 추가 시: 해당 품목 × H0~H6 행 7개를 추가하면 됨.

---

## HS 개정 타임라인

| 개정판 | 적용 연도 | UN Comtrade 코드 |
|--------|----------|-----------------|
| HS 1988/92 | ~1995 | H0 |
| HS 1996 | 1996~2001 | H1 |
| HS 2002 | 2002~2006 | H2 |
| HS 2007 | 2007~2011 | H3 |
| HS 2012 | 2012~2016 | H4 |
| HS 2017 | 2017~2021 | H5 |
| HS 2022 | 2022~     | H6 |

In [13]:
# 2026-05-15
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../..")
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
OUTPUT_PATH  = INTERIM_DIR / "반덤핑_HS코드_이력.csv"

## 1. 반덤핑 데이터에서 유니크 품목 추출

In [14]:
# regulation_events.csv에서 유니크 품목만 추출
# 동일 품목이 여러 국가에 규제될 경우 여러 행이 있으므로 drop_duplicates로 1개만 남김
# product_name_kr: 원문 품목명 (여러 표기가 있으면 첫 번째만 예시로 보존)
events_df = pd.read_csv(INTERIM_DIR / "regulation_events.csv")

unique_products = (
    events_df[["product_name_normalized", "product_name_kr"]]
    .drop_duplicates(subset="product_name_normalized")
    .rename(columns={"product_name_kr": "품목명_원문예시"})
    .rename(columns={"product_name_normalized": "품목명_정규화"})
    .reset_index(drop=True)
)

print(f"유니크 품목 수: {len(unique_products)}")
unique_products.head()

유니크 품목 수: 72


,품목명_정규화,품목명_원문예시
0,D.C.P,D.C.P
1,알루미나 시멘트,알루미나 시멘트
2,폴리아세탈수지,폴리아세탈수지
3,볼베어링,볼베어링
4,정제인산,정제인산


## 2. HS 개정판 × 품목 조합 생성 (이력 테이블 뼈대)

In [15]:
# HS 개정판 메타 정보
# 품목 1개당 이 7개 행이 생성됨
HS_REVISIONS = pd.DataFrame([
    {"hs_revision": "H0", "hs_version": "HS1992", "valid_from_year": 1988, "valid_to_year": 1995},
    {"hs_revision": "H1", "hs_version": "HS1996", "valid_from_year": 1996, "valid_to_year": 2001},
    {"hs_revision": "H2", "hs_version": "HS2002", "valid_from_year": 2002, "valid_to_year": 2006},
    {"hs_revision": "H3", "hs_version": "HS2007", "valid_from_year": 2007, "valid_to_year": 2011},
    {"hs_revision": "H4", "hs_version": "HS2012", "valid_from_year": 2012, "valid_to_year": 2016},
    {"hs_revision": "H5", "hs_version": "HS2017", "valid_from_year": 2017, "valid_to_year": 2021},
    {"hs_revision": "H6", "hs_version": "HS2022", "valid_from_year": 2022, "valid_to_year": 9999},
])

# cross join: 품목 N개 × 개정판 7개 = N×7 행
# 모든 품목에 대해 H0~H6 행을 각각 만들어 이력 기록 공간 확보
history_df = unique_products.merge(HS_REVISIONS, how="cross")

print(f"이력 테이블 크기: {history_df.shape}  ({len(unique_products)}품목 × 7개정판)")
history_df.head(14)  # 첫 번째 품목의 H0~H6 확인

이력 테이블 크기: (504, 6)  (72품목 × 7개정판)


,품목명_정규화,품목명_원문예시,hs_revision,hs_version,valid_from_year,valid_to_year
0,D.C.P,D.C.P,H0,HS1992,1988,1995
1,D.C.P,D.C.P,H1,HS1996,1996,2001
2,D.C.P,D.C.P,H2,HS2002,2002,2006
3,D.C.P,D.C.P,H3,HS2007,2007,2011
4,D.C.P,D.C.P,H4,HS2012,2012,2016
5,D.C.P,D.C.P,H5,HS2017,2017,2021
6,D.C.P,D.C.P,H6,HS2022,2022,9999
7,알루미나 시멘트,알루미나 시멘트,H0,HS1992,1988,1995
8,알루미나 시멘트,알루미나 시멘트,H1,HS1996,1996,2001
9,알루미나 시멘트,알루미나 시멘트,H2,HS2002,2002,2006


## 3. 기존 매핑 정보 병합

In [16]:
# product_hs_mapping.csv의 기존 매핑 정보를 이력 테이블에 붙임
# 현재 매핑은 개정판 구분 없이 1개 코드만 기록되어 있음
# → 일단 모든 개정판 행에 동일하게 붙이고, 이후 개정판별 변경 사항을 수동 수정
mapping_df = pd.read_csv(
    INTERIM_DIR / "product_hs_mapping.csv",
    dtype={"hs_code": str}  # HS 코드 앞자리 0 보존
)

history_df = history_df.merge(
    mapping_df[["product_name_normalized", "hs_code", "hs_level", "hs_description", "mapping_confidence", "mapping_note"]],
    left_on="품목명_정규화",
    right_on="product_name_normalized",
    how="left",
).drop(columns="product_name_normalized")

# 컬럼 이름 정리
history_df = history_df.rename(columns={
    "mapping_confidence": "mapping_status",
    "mapping_note": "note",
})

# hs_code가 없는 경우(needs_review) 빈 문자열로 처리
history_df["hs_code"] = history_df["hs_code"].fillna("").replace("nan", "")

print(f"병합 후 크기: {history_df.shape}")
history_df.head(7)

병합 후 크기: (504, 11)


,품목명_정규화,품목명_원문예시,hs_revision,hs_version,valid_from_year,valid_to_year,hs_code,hs_level,hs_description,mapping_status,note
0,D.C.P,D.C.P,H0,HS1992,1988,1995,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
1,D.C.P,D.C.P,H1,HS1996,1996,2001,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
2,D.C.P,D.C.P,H2,HS2002,2002,2006,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
3,D.C.P,D.C.P,H3,HS2007,2007,2011,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
4,D.C.P,D.C.P,H4,HS2012,2012,2016,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
5,D.C.P,D.C.P,H5,HS2017,2017,2021,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.
6,D.C.P,D.C.P,H6,HS2022,2022,9999,,NaN,NaN,needs_review,품목명-HS 코드 수동 매핑 필요.


## 4. 컬럼 순서 정리 및 저장

In [17]:
# 최종 컬럼 순서 정리
# is_changed_form_prev: 이전 개정판 대비 코드 변경 여부 (수동 기입 or 추후 파생)
# → 지금은 빈 컬럼으로만 생성해두고 나중에 채움
history_df["hs_code_changed_from_prev"] = ""  # 수동 기입 컬럼 (True/False/첫개정판은 '-')

FINAL_COLS = [
    "품목명_정규화",
    "품목명_원문예시",
    "hs_revision",       # H0~H6
    "hs_version",        # HS1992~HS2022
    "valid_from_year",   # 해당 개정판 적용 시작 연도
    "valid_to_year",     # 해당 개정판 적용 종료 연도
    "hs_code",           # HS 코드 (해당 개정판 기준)
    "hs_level",          # 코드 자리수 (4자리/6자리)
    "hs_description",    # HS 코드 설명
    "hs_code_changed_from_prev",  # 이전 개정판 대비 변경 여부
    "mapping_status",    # needs_review / low / confirmed
    "note",
]

history_df = history_df[FINAL_COLS]
history_df.head(14)

,품목명_정규화,품목명_원문예시,hs_revision,hs_version,valid_from_year,valid_to_year,hs_code,hs_level,hs_description,hs_code_changed_from_prev,mapping_status,note
0,D.C.P,D.C.P,H0,HS1992,1988,1995,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
1,D.C.P,D.C.P,H1,HS1996,1996,2001,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
2,D.C.P,D.C.P,H2,HS2002,2002,2006,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
3,D.C.P,D.C.P,H3,HS2007,2007,2011,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
4,D.C.P,D.C.P,H4,HS2012,2012,2016,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
5,D.C.P,D.C.P,H5,HS2017,2017,2021,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
6,D.C.P,D.C.P,H6,HS2022,2022,9999,,NaN,NaN,,needs_review,품목명-HS 코드 수동 매핑 필요.
7,알루미나 시멘트,알루미나 시멘트,H0,HS1992,1988,1995,2523,4.0,"Portland cement, aluminous cement and similar ...",,low,'시멘트' 키워드 기반 후보. 최종 분석 전 수동 검토 필요.
8,알루미나 시멘트,알루미나 시멘트,H1,HS1996,1996,2001,2523,4.0,"Portland cement, aluminous cement and similar ...",,low,'시멘트' 키워드 기반 후보. 최종 분석 전 수동 검토 필요.
9,알루미나 시멘트,알루미나 시멘트,H2,HS2002,2002,2006,2523,4.0,"Portland cement, aluminous cement and similar ...",,low,'시멘트' 키워드 기반 후보. 최종 분석 전 수동 검토 필요.


In [18]:
# 현황 요약
total_products = history_df["품목명_정규화"].nunique()
needs_review   = history_df[history_df["mapping_status"] == "needs_review"]["품목명_정규화"].nunique()
mapped         = total_products - needs_review

print(f"전체 품목 수       : {total_products}")
print(f"HS 코드 있음       : {mapped} ({mapped/total_products*100:.1f}%)")
print(f"수동 검토 필요     : {needs_review} ({needs_review/total_products*100:.1f}%)")
print(f"총 행 수 (×7개정판): {len(history_df)}")

전체 품목 수       : 72
HS 코드 있음       : 27 (37.5%)
수동 검토 필요     : 45 (62.5%)
총 행 수 (×7개정판): 504


In [19]:
# 저장
# encoding="utf-8-sig": 윈도우 엑셀에서 한글 깨짐 방지
history_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}")
print(f"\n--- 새 품목 추가 방법 ---")
print("이 CSV에 품목 × H0~H6 행 7개를 추가하고 hs_code 등을 직접 기입하면 됩니다.")

저장 완료: ../../data/interim/반덤핑_HS코드_이력.csv

--- 새 품목 추가 방법 ---
이 CSV에 품목 × H0~H6 행 7개를 추가하고 hs_code 등을 직접 기입하면 됩니다.
